# MediSafe — Model 1: Drug Interaction Risk Prediction (semi-supervised)

**Proposal objective 2 / AI Model 1:** *"A semi-supervised learning model that uses labeled and unlabeled drug
interaction data to classify potential interactions into predefined risk categories."*

| | |
|---|---|
| Data | DDInter 2.0 drug–drug interactions (Major / Moderate / Minor = labeled, Unknown = unlabeled) |
| Algorithm | `sklearn.semi_supervised.LabelSpreading` on a kNN graph of drug-pair feature vectors |
| Risk classes | Minor → **low**, Moderate → **moderate**, Major → **high** |
| Main metrics | macro-F1 and **recall of the high class** (missing a dangerous interaction is worse than a false alarm) |
| Baselines | majority class, Logistic Regression trained on labeled pairs only |

This notebook runs the whole pipeline step by step. It reuses the functions in `train_interaction.py`
(the script that writes `models/interaction_risk_model.json` for the backend), so the notebook and the
deployed model use exactly the same code.

**Run:** `cd model1 && .venv/bin/pip install -r requirements.txt jupyter && .venv/bin/python download_ddi.py`,
then open this notebook. A full run takes about 15–20 minutes (each kNN graph on 160k pairs takes ~2.5 min).

In [1]:
import json, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

import train_interaction as T          # same code as the deployed model
from drug_names import normalize, resolve, split_ingredients, pair_key

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 80)
CLASSES, SEED = T.CLASSES, T.SEED
RUN_LABEL_CURVE = True   # label-fraction experiment (~12 min); set False for a quick run
print("classes:", CLASSES)

classes: ['low', 'moderate', 'high']


## 1. Dataset: DDInter 2.0

Downloaded by `download_ddi.py` into `data/ddinter/` (8 CSV files, one per ATC group).
`SOURCE.json` records the download date, licence and a SHA-256 checksum of every file.

In [2]:
src = json.load(open("data/ddinter/SOURCE.json"))
print(src["dataset"], "|", src["license"], "| downloaded", src["downloaded_at_utc"][:10])
print(src["citation"])
pd.DataFrame([{"ATC": f["atc_code"], "file": f["file"], "rows": f["rows"], **f["levels"]} for f in src["files"]])

DDInter 2.0 | CC BY-NC-SA 4.0 | downloaded 2026-11-19
Tian Y. et al. DDInter 2.0. Nucleic Acids Research 53(D1):D1356, 2025. doi:10.1093/nar/gkae1004


,ATC,file,rows,Moderate,Major,Minor,Unknown
0,A,ddinter_downloads_code_A.csv,56367,33057,4906,3637,14767
1,B,ddinter_downloads_code_B.csv,15140,7359,2739,800,4242
2,D,ddinter_downloads_code_D.csv,25681,13519,2314,2015,7833
3,H,ddinter_downloads_code_H.csv,11727,7051,1203,707,2766
4,L,ddinter_downloads_code_L.csv,65389,39985,15915,2076,7413
5,P,ddinter_downloads_code_P.csv,5492,3270,1278,278,666
6,R,ddinter_downloads_code_R.csv,30563,19675,2617,1138,7133
7,V,ddinter_downloads_code_V.csv,12024,6451,2924,287,2362


In [3]:
pairs, names, files = T.load_pairs()
print(f"rows in files: {sum(f['rows'] for f in src['files']):,}  ->  unique unordered pairs: {len(pairs):,}")
pairs["Drug_A"] = pairs["id_a"].map(names); pairs["Drug_B"] = pairs["id_b"].map(names)
pairs[["Drug_A", "Drug_B", "Level", "y"]].head(8)

rows in files: 222,383  ->  unique unordered pairs: 160,235


,Drug_A,Drug_B,Level,y
0,Fenofibrate,Galactose,Unknown,-1
1,Quetiapine,Galactose,Unknown,-1
2,Doxazosin,Glipizide,Unknown,-1
3,Doxazosin,Glyburide,Unknown,-1
4,Misoprostol,Doxazosin,Unknown,-1
5,Doxazosin,Famotidine,Unknown,-1
6,Repaglinide,Doxazosin,Unknown,-1
7,Ondansetron,Doxazosin,Unknown,-1


The same pair can appear in several ATC files (A+B is listed under both drugs' groups), so pairs are made
order-independent and de-duplicated. No pair had two different severity levels.

In [4]:
y = pairs["y"].to_numpy()
labeled, unlabeled = np.where(y >= 0)[0], np.where(y < 0)[0]
counts = pairs["Level"].value_counts().reindex(["Major", "Moderate", "Minor", "Unknown"])
display(counts.to_frame("pairs").assign(share=lambda d: (d.pairs / d.pairs.sum()).round(3)))

fig, ax = plt.subplots(figsize=(6, 3))
counts.plot.bar(ax=ax, color=["#c0392b", "#e67e22", "#27ae60", "#95a5a6"])
ax.set_title("DDInter 2.0 pairs by severity level"); ax.set_ylabel("pairs"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
print(f"labeled: {len(labeled):,}   unlabeled (Unknown): {len(unlabeled):,}")

,pairs,share
Level,,
Major,26914,0.168
Moderate,96675,0.603
Minor,6833,0.043
Unknown,29813,0.186


labeled: 130,422   unlabeled (Unknown): 29,813


**Why semi-supervised?** 29,813 pairs (18.6 %) are known to interact but have no severity label. Label Spreading
uses them as unlabeled nodes in the graph, so their structure still helps the labeled ones.

The labeled data is **imbalanced** (moderate ≈ 74 %), which is why accuracy alone is misleading and we report
macro-F1 and per-class recall.

In [5]:
ids, ia, ib = T.drug_index(pairs)
deg = np.bincount(np.concatenate([ia, ib]), minlength=len(ids))
print(f"drugs: {len(ids):,}   median interactions per drug: {np.median(deg):.0f}   max: {deg.max():,}")
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(deg, bins=60, color="#3b6fb6"); ax.set_yscale("log")
ax.set_xlabel("interacting partners per drug"); ax.set_ylabel("drugs (log)"); ax.set_title("Degree distribution")
plt.tight_layout(); plt.show()

drugs: 1,939   median interactions per drug: 125   max: 910


## 2. Matching DDInter names to the Bangladeshi medicine catalog

The app's catalog (`../model2/medicine.csv`, MedEx) writes salts and INN names ("Amlodipine Besilate",
"Paracetamol"); DDInter uses base / US names ("Amlodipine", "Acetaminophen"). `drug_names.resolve()` handles
this (salt removal, two-way INN↔US synonyms). The same file is used by the backend.

In [6]:
known = {normalize(n) for n in names.values()}
examples = ["Paracetamol", "Amlodipine Besilate", "Salbutamol Sulfate", "Aspirin", "Warfarin Sodium",
            "Vitamin D3", "Tetracycline Hydrochloride (Topical)", "Gliclazide", "Etoricoxib"]
display(pd.DataFrame({"MedEx name": examples, "DDInter name": [resolve(e, known) for e in examples]}))

med = pd.read_csv("../model2/medicine.csv", usecols=["generic", "type"])
allo = med[med["type"] == "allopathic"]
full = allo["generic"].map(lambda g: all(resolve(i, known) for i in split_ingredients(str(g))))
print(f"allopathic products whose every ingredient matches DDInter: {full.mean():.1%} ({full.sum():,}/{len(allo):,})")
miss = pd.Series([i for g in allo.loc[~full, "generic"] for i in split_ingredients(str(g)) if not resolve(i, known)])
print("most common unmatched ingredients (not in DDInter):")
miss.value_counts().head(10)

,MedEx name,DDInter name
0,Paracetamol,acetaminophen
1,Amlodipine Besilate,amlodipine
2,Salbutamol Sulfate,salbutamol
3,Aspirin,acetylsalicylic acid
4,Warfarin Sodium,warfarin
5,Vitamin D3,cholecalciferol
6,Tetracycline Hydrochloride (Topical),tetracycline (topical)
7,Gliclazide,NaN
8,Etoricoxib,NaN


allopathic products whose every ingredient matches DDInter: 76.9% (16,438/21,363)
most common unmatched ingredients (not in DDInter):


Flucloxacillin Sodium       212
Domperidone Maleate         205
Vitamin B complex           165
Etoricoxib                  154
Vitamin B Complex           121
Vitamin B1, B6 & B12        120
Aceclofenac                 116
Ambroxol Hydrochloride      111
Tiemonium Methylsulphate    105
Zinc                        102
Name: count, dtype: int64

## 3. Train / validation / test split

Only labeled pairs are split (stratified 70 / 15 / 15). Validation picks the hyper-parameters; the test set is
used **once** at the end. During training, validation, test and Unknown pairs are all fed to Label Spreading
as unlabeled (label `-1`).

In [7]:
tr, rest = train_test_split(labeled, test_size=0.30, stratify=y[labeled], random_state=SEED)
va, te = train_test_split(rest, test_size=0.50, stratify=y[rest], random_state=SEED)
pd.DataFrame({s: pd.Series(y[idx]).map(dict(enumerate(CLASSES))).value_counts() for s, idx in
              [("train", tr), ("val", va), ("test", te)]}).loc[CLASSES]

,train,val,test
low,4783,1025,1025
moderate,67672,14501,14502
high,18840,4037,4037


## 4. Features

Every drug pair is one node. Its feature vector comes from the drug–drug interaction network:

* **drug embedding** — TruncatedSVD (32 dims) of the drug × drug adjacency matrix (*who interacts with whom*,
  no severity labels);
* **drug risk profile** — share of high / moderate / low among the drug's **training** pairs, with
  **leave-one-out** (a pair never sees its own label) and smoothing toward the class prior;
* **pair vector** — symmetric combination (sum, product, min, max, degrees), so A+B = B+A.

Only training labels are used to build features → no test-label leakage.

In [8]:
emb, logdeg = T.embeddings(len(ids), ia, ib)
y_train_only = np.full(len(y), -1); y_train_only[tr] = y[tr]
X, prior = T.build_features(ia, ib, len(ids), emb, logdeg, y_train_only)
print("feature matrix:", X.shape, "  class prior (train):", dict(zip(CLASSES, prior.round(3))))

feature matrix: (160235, 78)   class prior (train): {'low': np.float64(0.052), 'moderate': np.float64(0.741), 'high': np.float64(0.206)}


## 5. Label Spreading and hyper-parameter selection (validation set)

Grid: `alpha` (how much a node keeps its own label vs. its neighbours'), class-prior correction `beta`, and an
extra weight on the **high** score. (`train_interaction.py` also tries k = 20; k = 10 was better, so this
notebook builds only the k = 10 graph to save time.)

**Selection rule:** among settings within 0.01 of the best validation macro-F1, choose the one with the highest
validation **high-class recall**.

In [9]:
t = time.time()
graph = T.knn_graph(X, 10)
print(f"kNN graph: {X.shape[0]:,} nodes, {graph.nnz:,} edges ({time.time()-t:.0f}s)")

high = CLASSES.index("high"); rows = []
for alpha in T.GRID_ALPHA:
    dist, iters = T.spread(X, graph, y_train_only, alpha)
    for beta in T.GRID_BETA:
        for boost in T.GRID_BOOST:
            pv = T.correct(dist, prior, beta, boost)[va].argmax(1)
            rows.append(dict(alpha=alpha, beta=beta, high_boost=boost, iters=iters,
                             val_macro_f1=T.macro_f1(y[va], pv),
                             val_high_recall=((pv == high) & (y[va] == high)).sum() / (y[va] == high).sum()))
grid = pd.DataFrame(rows)
top = grid.val_macro_f1.max()
best = grid[grid.val_macro_f1 >= top - T.F1_TOLERANCE].sort_values(["val_high_recall", "val_macro_f1"]).iloc[-1]
print("best macro-F1 on val:", round(top, 4)); print("chosen:", best.to_dict())
grid.sort_values("val_macro_f1", ascending=False).head(10).round(4)

kNN graph: 160,235 nodes, 2,104,528 edges (158s)


best macro-F1 on val: 0.8878
chosen: {'alpha': 0.1, 'beta': 0.0, 'high_boost': 2.0, 'iters': 8.0, 'val_macro_f1': 0.8811168260029069, 'val_high_recall': 0.9145405003715631}


,alpha,beta,high_boost,iters,val_macro_f1,val_high_recall
1,0.1,0.0,1.25,8,0.8878,0.8635
0,0.1,0.0,1.00,8,0.8870,0.8328
2,0.1,0.0,1.50,8,0.8868,0.8861
16,0.3,0.0,1.25,14,0.8847,0.8573
17,0.3,0.0,1.50,14,0.8835,0.8786
15,0.3,0.0,1.00,14,0.8830,0.8229
3,0.1,0.0,2.00,8,0.8811,0.9145
18,0.3,0.0,2.00,14,0.8789,0.9118
31,0.6,0.0,1.25,31,0.8773,0.8333
32,0.6,0.0,1.50,31,0.8767,0.8628


## 6. Test-set evaluation (used once)

In [10]:
dist, _ = T.spread(X, graph, y_train_only, best.alpha)
raw = T.correct(dist, prior, best.beta, best.high_boost)
temp = T.fit_temperature(raw[va], y[va])
prob = T.temper(raw, temp)
pred = prob[te].argmax(1)
print(classification_report(y[te], pred, target_names=CLASSES, digits=3))

              precision    recall  f1-score   support

         low      0.911     0.755     0.826      1025
    moderate      0.964     0.940     0.952     14502
        high      0.816     0.922     0.866      4037

    accuracy                          0.927     19564
   macro avg      0.897     0.873     0.881     19564
weighted avg      0.930     0.927     0.928     19564



In [11]:
lr_pred = T.logreg(X, y_train_only, te)
majority = np.full(len(te), np.bincount(y[tr]).argmax())
def row(p):
    r = classification_report(y[te], p, target_names=CLASSES, output_dict=True, zero_division=0)
    return {"accuracy": accuracy_score(y[te], p), "macro-F1": T.macro_f1(y[te], p),
            "high recall": r["high"]["recall"], "high precision": r["high"]["precision"]}
pd.DataFrame({"Label Spreading (semi-supervised)": row(pred),
              "Logistic Regression (labeled only)": row(lr_pred),
              "Majority class": row(majority)}).T.round(3)

,accuracy,macro-F1,high recall,high precision
Label Spreading (semi-supervised),0.927,0.881,0.922,0.816
Logistic Regression (labeled only),0.807,0.713,0.846,0.650
Majority class,0.741,0.284,0.000,0.000


In [12]:
cm = confusion_matrix(y[te], pred)
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True (DDInter)"); ax.set_title("Label Spreading — test set")
for r in range(3):
    for c in range(3):
        ax.text(c, r, f"{cm[r, c]:,}", ha="center", va="center", color="white" if cm[r, c] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

**Reading the matrix:** most errors are between neighbouring classes (moderate ↔ high, low ↔ moderate);
high → low is rare. The high-class weight deliberately turns some moderate pairs into high (false alarms) to
miss fewer real Major interactions.

## 7. Confidence calibration

The backend shows a prediction only if its confidence ≥ 0.6, so the confidence should mean something.
A temperature is fitted on the validation set (log-loss) and checked with the **expected calibration error**
(ECE, 10 bins) and a reliability diagram on the test set.

In [13]:
print(f"temperature T = {temp}   test ECE before = {T.ece(raw[te], y[te])}   after = {T.ece(prob[te], y[te])}")
conf, ok = prob[te].max(1), prob[te].argmax(1) == y[te]
edges = np.linspace(0, 1, 11); mids, accs, ns = [], [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum() >= 20: mids.append(conf[m].mean()); accs.append(ok[m].mean()); ns.append(m.sum())
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect")
ax.plot(mids, accs, "o-", label="Label Spreading")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title("Reliability diagram (test)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"share of test predictions with confidence >= 0.6: {(conf >= 0.6).mean():.1%}")

temperature T = 1.0   test ECE before = 0.0219   after = 0.0219
share of test predictions with confidence >= 0.6: 92.9%


## 8. What does the unlabeled data add? (label-fraction experiment)

Keep only 5–100 % of the training labels (the rest become unlabeled), rebuild the features and graph, and compare
Label Spreading with Logistic Regression trained on the same labels. If the semi-supervised model stays ahead —
especially with few labels — the unlabeled pairs are helping.

In [14]:
if RUN_LABEL_CURVE:
    rng, curve = np.random.default_rng(SEED), []
    for frac in T.FRACTIONS:
        keep = tr if frac >= 1 else rng.choice(tr, int(len(tr) * frac), replace=False)
        yf = np.full(len(y), -1); yf[keep] = y[keep]
        Xf, pf = T.build_features(ia, ib, len(ids), emb, logdeg, yf)
        d, _ = T.spread(Xf, T.knn_graph(Xf, 10), yf, best.alpha)
        curve.append({"labels used": f"{frac:.0%}", "labeled pairs": len(keep),
                      "Label Spreading": T.macro_f1(y[te], T.correct(d, pf, best.beta, best.high_boost)[te].argmax(1)),
                      "Logistic Regression": T.macro_f1(y[te], T.logreg(Xf, yf, te))})
        print(curve[-1])
else:
    m = json.load(open("models/interaction_metrics.json"))
    curve = [{"labels used": f"{c['fraction']:.0%}", "labeled pairs": c["labeled_pairs"],
              "Label Spreading": c["label_spreading_macro_f1"], "Logistic Regression": c["logistic_regression_macro_f1"]}
             for c in m["label_fraction_curve"]]
curve = pd.DataFrame(curve)
ax = curve.plot(x="labels used", y=["Label Spreading", "Logistic Regression"], marker="o", figsize=(5.5, 3.5))
ax.set_ylabel("test macro-F1"); ax.set_title("Semi-supervised vs labeled-only"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
curve.round(3)

{'labels used': '5%', 'labeled pairs': 4564, 'Label Spreading': 0.6907471669851938, 'Logistic Regression': 0.5857969963408444}


{'labels used': '10%', 'labeled pairs': 9129, 'Label Spreading': 0.7595347367145463, 'Logistic Regression': 0.6337837693038093}


{'labels used': '25%', 'labeled pairs': 22823, 'Label Spreading': 0.8161643810723181, 'Logistic Regression': 0.6738429862426409}


{'labels used': '50%', 'labeled pairs': 45647, 'Label Spreading': 0.8605437466085615, 'Logistic Regression': 0.6972487386727266}


{'labels used': '100%', 'labeled pairs': 91295, 'Label Spreading': 0.8811544335724278, 'Logistic Regression': 0.7132885205168747}


,labels used,labeled pairs,Label Spreading,Logistic Regression
0,5%,4564,0.691,0.586
1,10%,9129,0.760,0.634
2,25%,22823,0.816,0.674
3,50%,45647,0.861,0.697
4,100%,91295,0.881,0.713


## 9. Final model: predict the Unknown pairs

For deployment the model is refit with **all** labels; the predictions for the 29,813 Unknown pairs are what the
backend shows as "Predicted (model estimate)". `train_interaction.py` writes exactly these predictions to
`models/interaction_risk_model.json`.

In [15]:
t = time.time()
Xa, pa = T.build_features(ia, ib, len(ids), emb, logdeg, y)
da, _ = T.spread(Xa, T.knn_graph(Xa, 10), y, best.alpha)
pall = T.temper(T.correct(da, pa, best.beta, best.high_boost), temp)
unk = pd.DataFrame({"Drug_A": pairs.loc[unlabeled, "Drug_A"].values, "Drug_B": pairs.loc[unlabeled, "Drug_B"].values,
                    "predicted": [CLASSES[i] for i in pall[unlabeled].argmax(1)],
                    "confidence": pall[unlabeled].max(1).round(3)})
print(f"final model refit in {time.time()-t:.0f}s")
display(unk["predicted"].value_counts().to_frame("Unknown pairs"))
print(f"shown in the app (confidence >= 0.6): {(unk.confidence >= 0.6).mean():.1%}")
unk.sort_values("confidence", ascending=False).groupby("predicted").head(3)

final model refit in 144s


,Unknown pairs
predicted,
moderate,21986
low,3944
high,3883


shown in the app (confidence >= 0.6): 86.5%


,Drug_A,Drug_B,predicted,confidence
21696,Rosiglitazone,Dicyclomine,moderate,1.0
21677,Lactulose,Dicyclomine,moderate,1.0
21672,Trazodone,Dicyclomine,moderate,1.0
21789,Ursodeoxycholic acid,Acyclovir,low,1.0
21589,Prednisone,Fentanyl,high,1.0
21585,Fentanyl,Hydrocortisone,high,1.0
21559,Triamcinolone,Fentanyl,high,1.0
21456,Diazepam,Hydrocortisone,low,1.0
21436,Tegaserod,Diazepam,low,1.0


In [16]:
# consistency check with the deployed JSON (written by train_interaction.py with the same settings)
dep = json.load(open("models/interaction_risk_model.json"))
same = np.mean([dep["predictions"].get(pair_key(normalize(a), normalize(b)), {}).get("category") == p
                for a, b, p in unk[["Drug_A", "Drug_B", "predicted"]].itertuples(index=False)])
print("deployed params:", dep["params"]); print(f"agreement with deployed predictions: {same:.1%}")

deployed params: {'k': 10, 'alpha': 0.1, 'beta': 0.0, 'high_boost': 2.0, 'val_macro_f1': 0.8811, 'val_high_recall': 0.9145, 'temperature': 1.0}
agreement with deployed predictions: 100.0%


## 10. Examples as the app shows them

* a pair with a **known** DDInter level → shown as *Verified* (no model needed);
* a pair whose level is **Unknown** → shown as *Predicted* with confidence;
* a pair **not in DDInter** → *Unknown / insufficient information* — never "safe".

In [17]:
lab = {pair_key(normalize(a), normalize(b)): lv for a, b, lv in pairs[["Drug_A", "Drug_B", "Level"]].itertuples(index=False)}
def lookup(x, y_):
    a, b = resolve(x, known), resolve(y_, known)
    if not a or not b: return "not in DDInter -> unknown (never 'safe')"
    k = pair_key(a, b); lv = lab.get(k)
    if lv is None: return "not listed -> unknown (never 'safe')"
    if lv != "Unknown": return f"Verified: DDInter {lv}"
    p = dep["predictions"][k]; return f"Predicted: {p['category']} (confidence {p['confidence']:.0%})"
for x, y_ in [("Warfarin Sodium", "Aspirin"), ("Paracetamol", "Warfarin Sodium"),
              ("Acetazolamide", "Amikacin"), ("Metformin Hydrochloride", "Gliclazide")]:
    print(f"{x} + {y_}: {lookup(x, y_)}")

Warfarin Sodium + Aspirin: Verified: DDInter Major
Paracetamol + Warfarin Sodium: Verified: DDInter Moderate
Acetazolamide + Amikacin: Predicted: high (confidence 96%)
Metformin Hydrochloride + Gliclazide: not in DDInter -> unknown (never 'safe')


## 11. Limitations

* **High-risk errors.** About 8 % of true Major pairs are still predicted moderate; about 18 % of pairs predicted
  high are not Major in DDInter (the deliberate false-alarm trade-off).
* **No ground truth for Unknown pairs.** Metrics are measured on labeled pairs; DDInter's Unknown pairs come mostly
  from a data-mining source, so accuracy on them cannot be measured and may be lower.
* **Coverage.** Only DDInter pairs are predicted. DDInter publishes 8 ATC groups, and some Bangladesh-specific
  generics (etoricoxib, gliclazide, domperidone, …) are not in it; 76.9 % of allopathic catalog products are fully
  matched.
* **Transductive model.** Label Spreading labels the nodes it was trained with; a brand-new pair needs a retrain.
* **Decision support only.** The model does not diagnose or prescribe; every result tells the user to consult a
  doctor or pharmacist.